In [1]:
# ============================================================
# CPTAC GBM SUBTYPE vs GTEx NORMAL
# PyDESeq2 differential expression analysis
#
# Comparisons:
#   Classical   vs Normal
#   Proneural   vs Normal
#   Mesenchymal vs Normal
#
# Input:
#   - CPTAC case-ID-based raw counts
#   - GTEx 7 normal frontal cortex raw counts
#   - CPTAC subtype metadata
#
# Output:
#   - One PyDESeq2 results CSV per subtype
# ============================================================


import pandas as pd
import numpy as np
from pathlib import Path

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


# ============================================================
# 1. PATHS
# ============================================================

cptac_file = Path(
    "../data/processed/CPTAC_GBM_tumor_raw_counts.csv"
)

gtex_file = Path(
    "../data/processed/GTEx_7_normal_frontal_cortex_raw_counts.csv"
)

subtype_file = Path(
    "../data/raw/all_subtypes.v5.1.tsv"
)

output_dir = Path(
    "../data/processed"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. LOAD CPTAC TUMOR COUNTS
# ============================================================

print("=" * 70)
print("LOADING CPTAC GBM TUMOR COUNTS")
print("=" * 70)

cptac = pd.read_csv(
    cptac_file
)

print("Shape:", cptac.shape)
print("First columns:")
print(cptac.columns[:10].tolist())


# ============================================================
# 3. LOAD GTEx CONTROLS
# ============================================================

print("\n" + "=" * 70)
print("LOADING GTEx CONTROLS")
print("=" * 70)

gtex = pd.read_csv(
    gtex_file
)

print("Shape:", gtex.shape)
print("Columns:")
print(gtex.columns.tolist())


# ============================================================
# 4. STANDARDIZE ENSEMBL IDS
# ============================================================

# CPTAC uses gene_id
cptac["gene_id"] = (
    cptac["gene_id"]
    .astype(str)
    .str.split(".")
    .str[0]
)

# GTEx uses Name
gtex["Name"] = (
    gtex["Name"]
    .astype(str)
    .str.split(".")
    .str[0]
)


# ============================================================
# 5. SET GENE ID AS INDEX
# ============================================================

cptac = cptac.set_index(
    "gene_id"
)

gtex = gtex.set_index(
    "Name"
)


# Remove GTEx description if present
if "Description" in gtex.columns:
    gtex = gtex.drop(
        columns="Description"
    )


# ============================================================
# 6. REMOVE DUPLICATE GENES
# ============================================================

cptac = cptac[
    ~cptac.index.duplicated(
        keep="first"
    )
]

gtex = gtex[
    ~gtex.index.duplicated(
        keep="first"
    )
]


# ============================================================
# 7. FIND COMMON GENES
# ============================================================

common_genes = (
    cptac.index
    .intersection(gtex.index)
)

print("\n" + "=" * 70)
print("GENE OVERLAP")
print("=" * 70)

print(
    "CPTAC genes:",
    len(cptac)
)

print(
    "GTEx genes:",
    len(gtex)
)

print(
    "Common genes:",
    len(common_genes)
)


# ============================================================
# 8. SUBSET TO COMMON GENES
# ============================================================

cptac_common = cptac.loc[
    common_genes
].copy()

gtex_common = gtex.loc[
    common_genes
].copy()


# ============================================================
# 9. STANDARDIZE SAMPLE IDS
# ============================================================

cptac_common.columns = (
    cptac_common.columns
    .astype(str)
    .str.strip()
)

gtex_common.columns = (
    gtex_common.columns
    .astype(str)
    .str.strip()
)


print("\nCPTAC samples:", cptac_common.shape[1])
print("GTEx controls:", gtex_common.shape[1])


# ============================================================
# 10. LOAD SUBTYPE METADATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING CPTAC SUBTYPE METADATA")
print("=" * 70)

subtypes = pd.read_csv(
    subtype_file,
    sep="\t"
)

print(
    "Metadata shape:",
    subtypes.shape
)

print(
    "\nMetadata columns:"
)

print(
    subtypes.columns.tolist()
)


# ============================================================
# 11. FIRST COLUMN = CPTAC CASE ID
# ============================================================

subtypes = subtypes.rename(
    columns={
        subtypes.columns[0]: "sample"
    }
)

subtypes["sample"] = (
    subtypes["sample"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 12. EXTRACT SUBTYPE INFORMATION
# ============================================================

subtype_table = subtypes[
    [
        "sample",
        "rna_wang_cancer_cell_2017"
    ]
].copy()

subtype_table = subtype_table.rename(
    columns={
        "rna_wang_cancer_cell_2017":
        "subtype"
    }
)


# ============================================================
# 13. CHECK CPTAC CASE-ID MATCHING
# ============================================================

matched_samples = set(
    cptac_common.columns
).intersection(
    subtype_table["sample"]
)

print("\n" + "=" * 70)
print("CPTAC CASE-ID MATCHING")
print("=" * 70)

print(
    "CPTAC count samples:",
    len(cptac_common.columns)
)

print(
    "Subtype metadata samples:",
    len(subtype_table)
)

print(
    "Matched samples:",
    len(matched_samples)
)


if len(matched_samples) == 0:

    raise ValueError(
        "No CPTAC samples matched the subtype metadata. "
        "The count matrix and subtype metadata are using "
        "different sample identifiers."
    )


# ============================================================
# 14. KEEP ONLY CPTAC SAMPLES WITH SUBTYPE INFORMATION
# ============================================================

subtype_table = subtype_table[
    subtype_table["sample"].isin(
        matched_samples
    )
].copy()


print("\nSubtype counts:")
print(
    subtype_table["subtype"]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# 15. DEFINE SUBTYPES
# ============================================================

subtype_names = [
    "Classical",
    "Proneural",
    "Mesenchymal"
]


# ============================================================
# 16. CHECK THAT EACH SUBTYPE EXISTS
# ============================================================

for subtype in subtype_names:

    n = (
        subtype_table["subtype"]
        .eq(subtype)
        .sum()
    )

    print(
        f"{subtype}: {n} samples"
    )

    if n == 0:

        raise ValueError(
            f"No samples found for subtype: {subtype}"
        )


# ============================================================
# 17. PREPARE RESULT STORAGE
# ============================================================

all_results = {}


# ============================================================
# 18. RUN PyDESeq2 FOR EACH SUBTYPE
# ============================================================

for subtype in subtype_names:

    print("\n")
    print("=" * 70)
    print(
        f"RUNNING PyDESeq2: {subtype} vs Normal"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Select CPTAC samples belonging to this subtype
    # --------------------------------------------------------

    subtype_samples = subtype_table.loc[
        subtype_table["subtype"] == subtype,
        "sample"
    ].tolist()


    print(
        f"{subtype} samples:",
        len(subtype_samples)
    )


    # --------------------------------------------------------
    # Combine subtype + GTEx controls
    # --------------------------------------------------------

    selected_samples = (
        subtype_samples
        + gtex_common.columns.tolist()
    )


    counts = pd.concat(
        [
            cptac_common[subtype_samples],
            gtex_common
        ],
        axis=1
    )


    # --------------------------------------------------------
    # Make sure counts are numeric
    # --------------------------------------------------------

    counts = counts.apply(
        pd.to_numeric,
        errors="coerce"
    )

    counts = counts.fillna(0)


    # --------------------------------------------------------
    # Raw counts must be integer
    # --------------------------------------------------------

    counts = (
        counts
        .round()
        .astype(int)
    )


    # --------------------------------------------------------
    # Remove genes with zero counts
    # --------------------------------------------------------

    counts = counts.loc[
        counts.sum(axis=1) > 0
    ]


    print(
        "Genes entering PyDESeq2:",
        counts.shape[0]
    )


    # --------------------------------------------------------
    # Create sample metadata
    # --------------------------------------------------------

    metadata = pd.DataFrame(
        index=counts.columns
    )

    metadata["condition"] = [
        subtype
        if sample in subtype_samples
        else "Normal"
        for sample in counts.columns
    ]


    print("\nSample groups:")

    print(
        metadata["condition"]
        .value_counts()
    )


    # --------------------------------------------------------
    # Explicitly make condition categorical
    # --------------------------------------------------------

    metadata["condition"] = pd.Categorical(
        metadata["condition"],
        categories=[
            "Normal",
            subtype
        ]
    )


    # --------------------------------------------------------
    # PyDESeq2 expects:
    #   rows    = samples
    #   columns = genes
    # --------------------------------------------------------

    counts_pydeseq = counts.T.copy()


    # Ensure metadata order matches count matrix
    metadata = metadata.loc[
        counts_pydeseq.index
    ]


    # --------------------------------------------------------
    # Create DESeqDataSet
    # --------------------------------------------------------

    dds = DeseqDataSet(
        counts=counts_pydeseq,
        metadata=metadata,
        design_factors="condition",
        refit_cooks=True
    )


    # --------------------------------------------------------
    # Run DESeq2
    # --------------------------------------------------------

    print("\nRunning DESeq2...")

    dds.deseq2()

    print(
        "DESeq2 finished."
    )


    # --------------------------------------------------------
    # Extract subtype vs Normal
    # --------------------------------------------------------

    stat_res = DeseqStats(
        dds,
        contrast=[
            "condition",
            subtype,
            "Normal"
        ]
    )


    stat_res.summary()


    # --------------------------------------------------------
    # Extract result dataframe
    # --------------------------------------------------------

    res = stat_res.results_df.copy()

    res = res.reset_index()


    # --------------------------------------------------------
    # Rename columns
    # --------------------------------------------------------

    res = res.rename(
        columns={
            "index": "ensembl_id",
            "baseMean": "baseMean",
            "log2FoldChange": "log2FC",
            "lfcSE": "SE",
            "stat": "statistic",
            "pvalue": "pvalue",
            "padj": "FDR"
        }
    )


    # --------------------------------------------------------
    # Standardize Ensembl IDs
    # --------------------------------------------------------

    res["ensembl_id"] = (
        res["ensembl_id"]
        .astype(str)
        .str.split(".")
        .str[0]
    )


    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    all_results[subtype] = res


# ============================================================
# 19. LOAD GENE NAME MAPPING
# ============================================================

print("\n" + "=" * 70)
print("PREPARING GENE NAME MAPPING")
print("=" * 70)


# IMPORTANT:
# `original_cptac` must be the ORIGINAL CPTAC file/dataframe
# containing BOTH:
#
#   gene_id
#   gene_name
#
# This is NOT CPTAC_GBM_tumor_raw_counts.csv if that processed
# file no longer contains gene_name.

original_cptac = pd.read_csv('../data/raw/CPTAC_GBM/0a14b390-558e-418a-83f8-148f459f2855/fdaad4b3-332b-4187-8e0d-c2f639add45c.rna_seq.augmented_star_gene_counts.tsv',skiprows=1,sep="\t")

gene_mapping = original_cptac[
    [
        "gene_id",
        "gene_name"
    ]
].copy()


gene_mapping["gene_id"] = (
    gene_mapping["gene_id"]
    .astype(str)
    .str.split(".")
    .str[0]
)


gene_mapping["gene_name"] = (
    gene_mapping["gene_name"]
    .astype(str)
    .str.strip()
)


gene_mapping = gene_mapping.drop_duplicates(
    subset="gene_id"
)


print(
    "Gene mappings:",
    len(gene_mapping)
)


# ============================================================
# 20. ADD GENE NAMES AND SAVE FINAL RESULTS
# ============================================================

final_results = {}


for subtype, res in all_results.items():

    print("\n" + "-" * 70)
    print(
        f"FINALIZING {subtype} RESULTS"
    )
    print("-" * 70)


    res = res.merge(
        gene_mapping,
        left_on="ensembl_id",
        right_on="gene_id",
        how="left"
    )


    res = res.drop(
        columns="gene_id"
    )


    # --------------------------------------------------------
    # Remove rows without a gene name
    # --------------------------------------------------------

    res["gene_name"] = (
        res["gene_name"]
        .replace(
            {
                "nan": np.nan,
                "": np.nan
            }
        )
    )


    # --------------------------------------------------------
    # Final column order
    # --------------------------------------------------------

    res = res[
        [
            "gene_name",
            "ensembl_id",
            "baseMean",
            "log2FC",
            "SE",
            "statistic",
            "pvalue",
            "FDR"
        ]
    ]


    # --------------------------------------------------------
    # Sort by FDR
    # --------------------------------------------------------

    res = res.sort_values(
        "FDR",
        na_position="last"
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    output_file = output_dir / (
        f"CPTAC_{subtype}"
        f"_vs_GTEx7_PyDESeq2_results.csv"
    )


    res.to_csv(
        output_file,
        index=False
    )


    final_results[subtype] = res


    print(
        "Genes:",
        len(res)
    )

    print(
        "Mapped gene names:",
        res["gene_name"].notna().sum()
    )

    print(
        "Saved:",
        output_file
    )


# ============================================================
# 21. SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("ANALYSIS COMPLETE")
print("=" * 70)

for subtype, res in final_results.items():

    significant = (
        res["FDR"] < 0.05
    ).sum()

    up = (
        (res["FDR"] < 0.05)
        & (res["log2FC"] > 1)
    ).sum()

    down = (
        (res["FDR"] < 0.05)
        & (res["log2FC"] < -1)
    ).sum()

    print(
        f"\n{subtype} vs Normal"
    )

    print(
        "Total genes:",
        len(res)
    )

    print(
        "FDR < 0.05:",
        significant
    )

    print(
        "Up (FDR < 0.05, log2FC > 1):",
        up
    )

    print(
        "Down (FDR < 0.05, log2FC < -1):",
        down
    )


# ============================================================
# 22. DISPLAY TOP RESULTS
# ============================================================

print("\n" + "=" * 70)
print("TOP CLASSICAL RESULTS")
print("=" * 70)

display(
    final_results["Classical"].head(20)
)

print("\n" + "=" * 70)
print("TOP PRONEURAL RESULTS")
print("=" * 70)

display(
    final_results["Proneural"].head(20)
)

print("\n" + "=" * 70)
print("TOP MESENCHYMAL RESULTS")
print("=" * 70)

display(
    final_results["Mesenchymal"].head(20)
)

LOADING CPTAC GBM TUMOR COUNTS
Shape: (60660, 100)
First columns:
['gene_id', 'C3L-00104', 'C3L-00365', 'C3L-00674', 'C3L-00677', 'C3L-01040', 'C3L-01043', 'C3L-01045', 'C3L-01046', 'C3L-01048']

LOADING GTEx CONTROLS
Shape: (56156, 9)
Columns:
['Name', 'Description', 'GTEX-NPJ7', 'GTEX-P44H', 'GTEX-Q2AG', 'GTEX-QVJO', 'GTEX-UTHO', 'GTEX-WVLH', 'GTEX-Y8DK']

GENE OVERLAP
CPTAC genes: 60616
GTEx genes: 56156
Common genes: 55617

CPTAC samples: 99
GTEx controls: 7

LOADING CPTAC SUBTYPE METADATA
Metadata shape: (109, 44)

Metadata columns:
['case', 'sample_type', 'multiomic', 'nmf_consensus', 'nmf_cluster_membership', 'rna_wang_cancer_cell_2017', 'mRNA_stemness_index', 'dna_methyl', 'dna_methyl_consensus', 'is_gcimp', 'mgmt_methyl_stp27_prediction', 'mgmt_methyl_stp27_probability', 'mgmt_methyl_clinical_test', 'mgmt_methyl_clinical_test_method', 'dna_methyl_dkfz_mnp_11b2', 'immune', 'xcell_immune_score', 'xcell_stroma_score', 'xcell_microenvironment_score', 'telomere', 'lipid', 'mirna', 

C:\Users\haina\AppData\Local\Temp\ipykernel_14412\165442000.py:515: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(
Fitting size factors...
... done in 0.06 seconds.

Fitting dispersions...
... done in 15.22 seconds.

Fitting dispersion trend curve...
... done in 1.00 seconds.

Fitting MAP dispersions...
... done in 15.45 seconds.

Fitting LFCs...
... done in 10.41 seconds.

Calculating cook's distance...
... done in 0.14 seconds.

Replacing 803 outlier genes.

Fitting dispersions...
... done in 0.25 seconds.

Fitting MAP dispersions...
... done in 0.28 seconds.

Fitting LFCs...
... done in 0.27 seconds.

Running Wald tests...


DESeq2 finished.


... done in 3.87 seconds.



Log2 fold change & Wald test p-value: condition Classical vs Normal
                    baseMean  log2FoldChange     lfcSE       stat  \
ENSG00000000003  1448.557248        1.432102  0.217926   6.571506   
ENSG00000000005    17.511258        1.194952  0.399035   2.994605   
ENSG00000000419   892.913320       -0.441637  0.143441  -3.078868   
ENSG00000000457   804.655806        0.777751  0.087756   8.862673   
ENSG00000000460   479.922472        2.246662  0.155725  14.427153   
...                      ...             ...       ...        ...   
ENSG00000284587     0.085896       -1.178427  3.653268  -0.322568   
ENSG00000284591     0.146898        0.632821  2.848682   0.222145   
ENSG00000284592     0.037931        0.171154  3.776860   0.045316   
ENSG00000284595     0.109268       -1.488929  2.792227  -0.533241   
ENSG00000284600     6.643872       -1.348198  0.608789  -2.214558   

                       pvalue          padj  
ENSG00000000003  4.980878e-11  1.884637e-10  
ENSG0000000

...

,gene_name,ensembl_id,baseMean,log2FC,SE,statistic,pvalue,FDR
8512,NBEAL1,ENSG00000144426,2208.236974,5.901682,0.140550,41.990007,0.000000e+00,0.000000e+00
15173,NGRN,ENSG00000182768,7542.332704,-4.863749,0.099390,-48.936088,0.000000e+00,0.000000e+00
12180,BSCL2,ENSG00000168000,3792.203201,-7.337624,0.144487,-50.783968,0.000000e+00,0.000000e+00
1318,BCAP29,ENSG00000075790,1408.177104,-5.588855,0.116747,-47.871540,0.000000e+00,0.000000e+00
15923,ATP6V0C,ENSG00000185883,3655.067087,-10.386180,0.207160,-50.135996,0.000000e+00,0.000000e+00
15482,VPS33B,ENSG00000184056,388.176352,-3.581081,0.082572,-43.369203,0.000000e+00,0.000000e+00
23109,PPP3R1,ENSG00000221823,4675.433738,-4.625658,0.112659,-41.059056,0.000000e+00,0.000000e+00
21429,SUPT4H1,ENSG00000213246,490.181964,-10.361287,0.251008,-41.278634,0.000000e+00,0.000000e+00
15049,ARL6IP4,ENSG00000182196,984.339618,-7.920596,0.186974,-42.362029,0.000000e+00,0.000000e+00
5536,RPS10,ENSG00000124614,1917.600723,-6.132129,0.145933,-42.020181,0.000000e+00,0.000000e+00



TOP PRONEURAL RESULTS


,gene_name,ensembl_id,baseMean,log2FC,SE,statistic,pvalue,FDR
35948,BCKDHA,ENSG00000248098,278.406210,-6.493448,0.151265,-42.927570,0.000000e+00,0.000000e+00
21618,SUPT4H1,ENSG00000213246,445.090331,-10.245221,0.237265,-43.180526,0.000000e+00,0.000000e+00
1319,BCAP29,ENSG00000075790,1268.193171,-5.969173,0.147224,-40.544912,0.000000e+00,0.000000e+00
10442,ST6GALNAC6,ENSG00000160408,2066.365199,-7.476027,0.161314,-46.344428,0.000000e+00,0.000000e+00
15092,ARL6IP4,ENSG00000182196,895.129855,-7.924095,0.186996,-42.375822,0.000000e+00,0.000000e+00
12723,GABARAP,ENSG00000170296,1947.669616,-7.945987,0.163181,-48.694364,0.000000e+00,0.000000e+00
5541,RPS10,ENSG00000124614,1749.694635,-6.192655,0.144022,-42.998022,0.000000e+00,0.000000e+00
4851,CYP20A1,ENSG00000119004,1830.120011,3.963561,0.105101,37.711840,0.000000e+00,0.000000e+00
22967,RNASEK,ENSG00000219200,1049.700539,-6.350304,0.152052,-41.763891,0.000000e+00,0.000000e+00
15973,ATP6V0C,ENSG00000185883,3318.996102,-10.201687,0.224874,-45.366232,0.000000e+00,0.000000e+00



TOP MESENCHYMAL RESULTS


,gene_name,ensembl_id,baseMean,log2FC,SE,statistic,pvalue,FDR
12717,GABARAP,ENSG00000170296,1581.309514,-7.893875,0.193308,-40.835826,0.000000e+00,0.000000e+00
45496,RPL17,ENSG00000265681,1803.739349,-6.294266,0.164078,-38.361354,0.000000e+00,0.000000e+00
5750,DYNLRB1,ENSG00000125971,1512.760717,-4.649732,0.108824,-42.726912,0.000000e+00,0.000000e+00
3143,PIK3R2,ENSG00000105647,751.294952,-8.461434,0.163683,-51.693998,0.000000e+00,0.000000e+00
46532,FDX2,ENSG00000267673,234.159326,-9.080348,0.240432,-37.766744,0.000000e+00,0.000000e+00
23356,PPP3R1,ENSG00000221823,3618.729425,-4.726026,0.082281,-57.437443,0.000000e+00,0.000000e+00
5353,PFDN5,ENSG00000123349,1576.931897,-5.364345,0.141353,-37.950087,0.000000e+00,0.000000e+00
5540,RPS10,ENSG00000124614,1437.785007,-6.187776,0.128350,-48.210356,0.000000e+00,0.000000e+00
12191,BSCL2,ENSG00000168000,2799.116966,-7.471254,0.185778,-40.215939,0.000000e+00,0.000000e+00
13446,FKBP2,ENSG00000173486,295.949464,-7.400044,0.175849,-42.081822,0.000000e+00,0.000000e+00


In [2]:
original_cptac = pd.read_csv('../data/raw/CPTAC_GBM/0a14b390-558e-418a-83f8-148f459f2855/fdaad4b3-332b-4187-8e0d-c2f639add45c.rna_seq.augmented_star_gene_counts.tsv',skiprows=1)
original_cptac.columns

Index(['gene_id\tgene_name\tgene_type\tunstranded\tstranded_first\tstranded_second\ttpm_unstranded\tfpkm_unstranded\tfpkm_uq_unstranded'], dtype='object')